# Stanford RNA 3D Folding Part 2 - Competitive Baseline v1

这个 notebook 从“可提交流程”升级到“可验证、可提分”的第一版基线：

- 使用 `train_sequences + train_labels` 构建模板库
- 对每个 `test` target 做序列相似模板检索
- 将模板坐标重采样到目标长度，生成 5 组候选结构
- 在 `validation` 上给出一个 best-of-5 的对齐 RMSD 指标（离线自检）
- 输出 `submission.csv`

> 这是实战起点版本：重点是稳定性、可解释和可持续迭代。

In [ ]:
import os
import re
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

DATA_DIR = "/kaggle/input/stanford-rna-3d-folding-2"

NUC_SET = set("ACGU")

def normalize_sequence(seq: str) -> str:
    s = (seq or "").upper().replace("T", "U")
    return "".join(ch if ch in NUC_SET else "A" for ch in s)


def infer_target_id_and_resid(labels: pd.DataFrame) -> pd.DataFrame:
    out = labels.copy()
    if "target_id" not in out.columns:
        out["target_id"] = out["ID"].astype(str).str.rsplit("_", n=1).str[0]
    if "resid" not in out.columns:
        out["resid"] = out["ID"].astype(str).str.rsplit("_", n=1).str[1].astype(int)
    return out


def discover_xyz_triplets(columns):
    triplets = []
    x_cols = [c for c in columns if re.fullmatch(r"x_\d+", c)]
    for x in sorted(x_cols, key=lambda s: int(s.split("_")[1])):
        i = x.split("_")[1]
        y = f"y_{i}"
        z = f"z_{i}"
        if y in columns and z in columns:
            triplets.append((x, y, z))
    return triplets


def average_available_xyz(row: pd.Series, triplets):
    coords = []
    for x, y, z in triplets:
        vals = [row.get(x, np.nan), row.get(y, np.nan), row.get(z, np.nan)]
        if np.all(np.isfinite(vals)):
            coords.append(vals)
    if not coords:
        return np.array([np.nan, np.nan, np.nan], dtype=np.float32)
    return np.asarray(coords, dtype=np.float32).mean(axis=0)


@dataclass
class Template:
    target_id: str
    sequence: str
    coords: np.ndarray
    k3: set


def kmer_set(seq: str, k: int = 3):
    if len(seq) < k:
        return {seq}
    return {seq[i : i + k] for i in range(len(seq) - k + 1)}


def sampled_identity(s1: str, s2: str, n_samples: int = 128) -> float:
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    m = max(8, min(n_samples, len(s1), len(s2)))
    idx1 = np.linspace(0, len(s1) - 1, m).round().astype(int)
    idx2 = np.linspace(0, len(s2) - 1, m).round().astype(int)
    return float(np.mean([s1[i] == s2[j] for i, j in zip(idx1, idx2)]))


def sequence_similarity(query: str, tpl: Template) -> float:
    qk3 = kmer_set(query, k=3)
    inter = len(qk3 & tpl.k3)
    union = max(1, len(qk3 | tpl.k3))
    jaccard = inter / union

    ident = sampled_identity(query, tpl.sequence)
    len_ratio = min(len(query), len(tpl.sequence)) / max(len(query), len(tpl.sequence))

    # 权重偏向于 k-mer overlap，同时保留长度和点采样 identity
    return 0.55 * jaccard + 0.30 * ident + 0.15 * len_ratio


def resample_coords(coords: np.ndarray, new_len: int) -> np.ndarray:
    old_len = len(coords)
    if old_len == new_len:
        return coords.copy()
    if old_len == 1:
        return np.repeat(coords, new_len, axis=0)

    old_x = np.linspace(0.0, 1.0, old_len)
    new_x = np.linspace(0.0, 1.0, new_len)
    out = np.zeros((new_len, 3), dtype=np.float32)
    for d in range(3):
        out[:, d] = np.interp(new_x, old_x, coords[:, d])
    return out


def diversify(coords: np.ndarray, model_idx: int) -> np.ndarray:
    n = len(coords)
    i = np.arange(n, dtype=np.float32)
    wobble = 0.10 + 0.05 * model_idx
    wave = np.stack(
        [
            np.sin(0.23 * i + 0.8 * model_idx),
            np.cos(0.19 * i + 0.5 * model_idx),
            np.sin(0.17 * i + 1.1 * model_idx),
        ],
        axis=1,
    ).astype(np.float32)
    scale = 1.0 + 0.01 * model_idx
    return coords * scale + wobble * wave


def make_helix(length: int, model_idx: int) -> np.ndarray:
    i = np.arange(length, dtype=np.float32)
    radius = 8.0 + 0.6 * model_idx
    theta = 0.55 + 0.02 * model_idx
    phase = 0.7 * model_idx
    rise = 2.4 + 0.05 * model_idx
    x = radius * np.cos(theta * i + phase)
    y = radius * np.sin(theta * i + phase)
    z = rise * i
    return np.stack([x, y, z], axis=1).astype(np.float32)


def build_templates(seq_df: pd.DataFrame, labels_df: pd.DataFrame):
    seq_map = {
        row.target_id: normalize_sequence(row.sequence)
        for row in seq_df.itertuples(index=False)
    }

    labels_df = infer_target_id_and_resid(labels_df)
    triplets = discover_xyz_triplets(labels_df.columns)
    if not triplets:
        raise ValueError("labels 中未发现 x_i/y_i/z_i 坐标列")

    templates = []
    for tid, g in labels_df.groupby("target_id"):
        if tid not in seq_map:
            continue

        g2 = g.sort_values("resid")
        coords = np.stack([average_available_xyz(r, triplets) for _, r in g2.iterrows()], axis=0)
        valid = np.isfinite(coords).all(axis=1)
        coords = coords[valid]
        if len(coords) < 8:
            continue

        # 去掉平移项，保留形状
        coords = coords - coords.mean(axis=0, keepdims=True)

        seq = seq_map[tid]
        templates.append(Template(target_id=tid, sequence=seq, coords=coords.astype(np.float32), k3=kmer_set(seq, 3)))

    return templates


def predict_five_models(query_seq: str, templates, top_k: int = 5):
    query_seq = normalize_sequence(query_seq)
    n = len(query_seq)

    if not templates:
        return np.stack([make_helix(n, i) for i in range(5)], axis=0)

    scored = [(sequence_similarity(query_seq, tpl), tpl) for tpl in templates]
    scored.sort(key=lambda x: x[0], reverse=True)

    chosen = [tpl for _, tpl in scored[:top_k]]
    preds = []
    for i, tpl in enumerate(chosen):
        base = resample_coords(tpl.coords, n)
        preds.append(diversify(base, i))

    # 不足 5 个候选时，用最佳模板的扰动补齐
    while len(preds) < 5:
        base = preds[0] if preds else make_helix(n, 0)
        preds.append(diversify(base, len(preds)))

    return np.stack(preds[:5], axis=0).astype(np.float32)


def kabsch_aligned_rmsd(pred: np.ndarray, true: np.ndarray) -> float:
    pred = pred.astype(np.float64)
    true = true.astype(np.float64)

    pred_c = pred - pred.mean(axis=0, keepdims=True)
    true_c = true - true.mean(axis=0, keepdims=True)

    h = pred_c.T @ true_c
    u, _, vt = np.linalg.svd(h)
    r = vt.T @ u.T
    if np.linalg.det(r) < 0:
        vt[-1, :] *= -1
        r = vt.T @ u.T

    pred_rot = pred_c @ r
    diff = pred_rot - true_c
    return float(np.sqrt((diff * diff).sum() / len(pred)))


def labels_to_target_coords(labels_df: pd.DataFrame):
    labels_df = infer_target_id_and_resid(labels_df)
    triplets = discover_xyz_triplets(labels_df.columns)

    out = {}
    for tid, g in labels_df.groupby("target_id"):
        g2 = g.sort_values("resid")
        coords = np.stack([average_available_xyz(r, triplets) for _, r in g2.iterrows()], axis=0)
        valid = np.isfinite(coords).all(axis=1)
        coords = coords[valid]
        if len(coords) >= 8:
            out[tid] = coords.astype(np.float32)
    return out


start = time.time()

required_files = [
    "train_sequences.csv",
    "train_labels.csv",
    "validation_sequences.csv",
    "validation_labels.csv",
    "test_sequences.csv",
]
for fn in required_files:
    p = os.path.join(DATA_DIR, fn)
    if not os.path.exists(p):
        raise FileNotFoundError(f"未找到 {p}，请确认已在 Kaggle Notebook 添加正确比赛数据")

train_sequences = pd.read_csv(os.path.join(DATA_DIR, "train_sequences.csv"))
train_labels = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))
val_sequences = pd.read_csv(os.path.join(DATA_DIR, "validation_sequences.csv"))
val_labels = pd.read_csv(os.path.join(DATA_DIR, "validation_labels.csv"))
test_sequences = pd.read_csv(os.path.join(DATA_DIR, "test_sequences.csv"))

print("train_sequences:", train_sequences.shape)
print("train_labels:", train_labels.shape)
print("validation_sequences:", val_sequences.shape)
print("validation_labels:", val_labels.shape)
print("test_sequences:", test_sequences.shape)

# ---------- 构建模板库（仅用 train，防止信息泄露） ----------
templates = build_templates(train_sequences, train_labels)
print(f"templates built: {len(templates)}")

# ---------- 离线验证（best-of-5 aligned RMSD） ----------
val_truth = labels_to_target_coords(val_labels)
val_seq_map = {
    row.target_id: normalize_sequence(row.sequence)
    for row in val_sequences.itertuples(index=False)
}

rmsd_list = []
for tid, true_coords in val_truth.items():
    if tid not in val_seq_map:
        continue

    preds5 = predict_five_models(val_seq_map[tid], templates, top_k=5)
    # 预测长度和标签长度可能有偏差，截断到共同长度
    m = min(len(true_coords), preds5.shape[1])
    true_c = true_coords[:m]

    best = min(kabsch_aligned_rmsd(preds5[k, :m], true_c) for k in range(5))
    rmsd_list.append(best)

if rmsd_list:
    print(f"Validation best-of-5 aligned RMSD | mean={np.mean(rmsd_list):.4f}, median={np.median(rmsd_list):.4f}, n={len(rmsd_list)}")
else:
    print("Validation 评估未得到有效样本，请检查数据。")

# ---------- 生成 submission.csv ----------
rows = []
for row in test_sequences.itertuples(index=False):
    target_id = row.target_id
    seq = normalize_sequence(row.sequence)

    preds5 = predict_five_models(seq, templates, top_k=5)  # (5, n, 3)

    for resid, resname in enumerate(seq, start=1):
        rec = {
            "ID": f"{target_id}_{resid}",
            "resname": resname,
            "resid": resid,
        }
        for k in range(5):
            rec[f"x_{k+1}"] = float(preds5[k, resid - 1, 0])
            rec[f"y_{k+1}"] = float(preds5[k, resid - 1, 1])
            rec[f"z_{k+1}"] = float(preds5[k, resid - 1, 2])
        rows.append(rec)

submission = pd.DataFrame(rows)
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]
submission[coord_cols] = submission[coord_cols].clip(lower=-999.999, upper=9999.999)

submission.to_csv("submission.csv", index=False)
print("submission saved:", submission.shape, "-> submission.csv")
print(f"elapsed: {time.time() - start:.1f}s")
display(submission.head(10))
